# Tvorba aplikácií na generovanie obrázkov (OpenAI)

S veľkými jazykovými modelmi (LLM) sú spojené nielen textové generácie. Môžete tiež generovať obrázky z textových popisov. Obrázky ako modalita sú užitočné v oblasti zdravotníctva, architektúry, turizmu, vývoja hier, marketingu a ďalších. V tejto lekcii pracujeme s dnešnými **GPT Image** modelmi na platforme OpenAI.

## Ciele učenia

Na konci tejto lekcie budete vedieť:

- Vysvetliť, čo je generovanie obrázkov a kde je užitočné.
- Pochopiť rodinu modelov `gpt-image` a ako sa líši od starších modelov DALL·E.
- Vytvoriť aplikáciu na generovanie obrázkov a upravovať obrázky.

## Čo je generovanie obrázkov?

Modely na generovanie obrázkov vytvárajú obrázky z textovej výzvy. Moderné modely ako `gpt-image` sa počas tréningu učia vzťah medzi textom a obrázkami, potom postupne premieňajú náhodný šum na obrázok zodpovedajúci vašej výzve.

Dve dobre známe rodiny modelov na obrázky sú:

- **`gpt-image` (OpenAI)** — súčasná generácia použitá v tejto lekcii. Podporuje generovanie obrázkov z textu a úpravu obrázkov (do­maľovanie pomocou štítku).
- **Midjourney** — populárny model tretích strán s vlastnou službou a pracovným postupom založeným na Discorde.

> Staršie OpenAI modely obrázkov — **DALL·E 2** a **DALL·E 3** — sú zastarané. DALL·E 3 už nie je dostupný pre nové nasadenia a funkcie ako `create_variation` existovali iba v DALL·E 2. Pre nové aplikácie používajte modely `gpt-image`.

> **Dôležité:** Modely `gpt-image` vracajú vygenerovaný obrázok ako **base64** (`b64_json`), nie ako URL. Váš kód dekóduje base64 reťazec do bajtov a uloží ho — neexistuje URL na stiahnutie obrázka.


## Vytvorenie vašej prvej aplikácie na generovanie obrázkov

Čo teda potrebujete na vytvorenie aplikácie na generovanie obrázkov? Potrebujete nasledujúce knižnice:

- **python-dotenv**, veľmi sa odporúča používať túto knižnicu na ukladanie vašich tajomstiev v súbore *.env* mimo kódu.
- **openai**, táto knižnica je to, čo použijete na interakciu s OpenAI API.
- **pillow**, na prácu s obrázkami v Pythone.


1. Vytvorte súbor *.env* s nasledujúcim obsahom:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Zozbierte vyššie uvedené knižnice do súboru s názvom *requirements.txt* takto:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ďalej vytvorte virtuálne prostredie a nainštalujte knižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pre Windows použite nasledujúce príkazy na vytvorenie a aktiváciu vášho virtuálneho prostredia:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Pridajte nasledovný kód do súboru s názvom *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Poďme si tento kód vysvetliť:

- Najprv importujeme knižnice, ktoré potrebujeme, vrátane knižnice OpenAI, knižnice dotenv, modulu base64 a knižnice Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Potom vytvoríme klienta, ktorý načíta API kľúč z vášho ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Ďalej generujeme obrázok:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modely vracajú obrázok ako **base64** reťazec v `data[0].b64_json`. Dekódujeme ho do bajtov a zapíšeme do súboru — neexistuje URL na stiahnutie.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Nakoniec otvoríme obrázok a použijeme štandardný prehliadač obrázkov na jeho zobrazenie:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Viac detailov o generovaní obrázka

Pozrime sa na parametre `images.generate`:

- **model**, je obrazový model, ktorý sa použije (napríklad `gpt-image-2`).
- **prompt**, je textový prompt použitý na generovanie obrázka. Tu je to "Zajko na koni, drží lízatko, na hmlistej lúke kde rastú narcisky".
- **size**, je veľkosť generovaného obrázka (napríklad `1024x1024`, `1536x1024`, `1024x1536` alebo `"auto"`).
- **n**, je počet generovaných obrázkov. Tu generujeme jeden.

> Obrazové modely neberú parameter `temperature` — to je riadenie generovania textu. Ak chcete získať rôznorodosť, zavolajte API znova; ak chcete znížiť rôznorodosť, spravte váš prompt konkrétnejším.

## Ďalšie možnosti generovania obrázkov

Videli ste, ako generovať obrázok pomocou niekoľkých riadkov Pythonu. `gpt-image` modely môžu tiež **editovať** existujúci obrázok. Poskytnutím obrázka, voliteľnej **masky** (ktorá označuje oblasť na zmenu) a promptu môžete upraviť časť obrázka — napríklad pridať klobúk nášmu zajkovi.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Základný obrázok obsahuje iba zajaca; finálny obrázok má na zajacovi klobúk.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
